# 02 — Lastik Degradasyon Modeli
### Pacejka Magic Formula + Termal Dinamik + Undercut Analizi

**Bu notebook'ta:**
- Pacejka Magic Formula — bileşenler ve katsayılar
- Termal ODE simülasyonu (ısınma/soğuma)
- Wear cliff tespiti ve stint stratejisi
- Compound karşılaştırması (Soft/Medium/Hard)
- Undercut/Overcut fırsat analizi

---


## 1. Pacejka Magic Formula

Grip katsayısı $\mu$, slip enerjisinin bir fonksiyonudur:

$$\mu(x) = D \cdot \sin\left(C \cdot \arctan\left(Bx - E(Bx - \arctan(Bx))\right)\right)$$

Parametreler:
- $B$ (stiffness): eğrinin başlangıç eğimi
- $C$ (shape): tepe noktasının genişliği
- $D$ (peak): maksimum grip değeri
- $E$ (curvature): tepe sonrası düşüş şekli

**Wear degradasyon** (kümülatif enerji ile):
$$\mu_{wear}(E_{cum}) = e^{-\lambda \cdot E_{cum}}$$

**Cliff sonrası** (threshold aşıldığında):
$$\mu_{cliff} = e^{-\lambda \cdot E_{cliff}} \cdot e^{-\lambda \cdot \kappa \cdot (E - E_{cliff})}$$


In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pitstop.simulation.tire_model import (
    COMPOUNDS, pacejka_grip, thermal_penalty, wear_multiplier,
    simulate_stint, undercut_delta
)

# Tüm compound'lar için Pacejka eğrisi
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Pacejka Magic Formula — Compound Karşılaştırması', fontsize=13, fontweight='bold')

x = np.linspace(0, 2.0, 300)
compound_colors = {'soft': '#E74C3C', 'medium': '#F39C12', 'hard': '#95A5A6'}

for ax, (param_name, param_func, xlabel) in zip(axes, [
    ('Grip vs Slip Enerjisi', lambda c, xi: pacejka_grip(c, xi), 'Normalised slip energy'),
    ('Termal Penaltı', lambda c, theta: thermal_penalty(c, theta), 'Sıcaklık (°C)'),
    ('Wear Faktörü', lambda c, e: wear_multiplier(c, e), 'Kümülatif enerji (kJ/m)'),
]):
    for comp_name in ['soft', 'medium', 'hard']:
        c = COMPOUNDS[comp_name]
        if 'Slip' in param_name:
            y = [param_func(c, xi) for xi in x]
            xi_range = x
        elif 'Termal' in param_name:
            xi_range = np.linspace(20, 130, 300)
            y = [param_func(c, t) for t in xi_range]
        else:
            xi_range = np.linspace(0, 600, 300)
            y = [param_func(c, e) for e in xi_range]
            ax.axvline(c.wear_cliff, color=compound_colors[comp_name],
                      linestyle=':', alpha=0.5, label=f'{comp_name} cliff')
        ax.plot(xi_range, y, color=compound_colors[comp_name],
                linewidth=2.5, label=comp_name.capitalize())
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Faktör')
    ax.set_title(param_name)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_pacejka_components.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Stint Simülasyonu — Bileşik Degradasyon

Her turda total grip:
$$\mu_{eff}(E_{cum}, \theta) = \mu_{base}(x) \cdot \mu_{thermal}(\theta) \cdot \mu_{wear}(E_{cum})$$

Tur süresi artışı:
$$\Delta t_{lap} = k_{deg} \cdot n_{age}^{0.7} \quad (\text{cliff öncesi})$$
$$\Delta t_{lap} \times \kappa_{cliff} \quad (\text{cliff sonrası})$$


In [ ]:
# 3 compound için 45-tur stint simülasyonu
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Lastik Stint Simülasyonu — Tüm Compound\'lar', fontsize=13, fontweight='bold')

compound_data = {}
for comp in ['soft', 'medium', 'hard']:
    df = simulate_stint(comp, n_laps=45, track_temp=42, ambient_temp=24)
    compound_data[comp] = df

colors = {'soft': '#E74C3C', 'medium': '#F39C12', 'hard': '#95A5A6'}

# Tur süresi artışı
ax = axes[0, 0]
for comp, df in compound_data.items():
    ax.plot(df['lap'], df['lap_delta'], color=colors[comp], linewidth=2.5, label=comp.capitalize())
    cliff_laps = df[df['past_cliff']]['lap']
    if not cliff_laps.empty:
        ax.axvline(cliff_laps.min(), color=colors[comp], linestyle='--', alpha=0.5, linewidth=1.5)
        ax.text(cliff_laps.min()+0.5, df[df['lap']==cliff_laps.min()]['lap_delta'].values[0],
               f'{comp[:1].upper()} cliff', fontsize=7, color=colors[comp])
ax.set_xlabel('Tur (lastik yaşı)')
ax.set_ylabel('Tur süresi artışı (s)')
ax.set_title('Degradasyon — Tur Süresi Artışı')
ax.legend()
ax.grid(alpha=0.3)

# Sıcaklık profili
ax = axes[0, 1]
for comp, df in compound_data.items():
    ax.plot(df['lap'], df['temperature'], color=colors[comp], linewidth=2.5, label=comp.capitalize())
    ax.axhline(COMPOUNDS[comp].theta_optimal, color=colors[comp], linestyle=':', alpha=0.4)
ax.set_xlabel('Tur')
ax.set_ylabel('Lastik sıcaklığı (°C)')
ax.set_title('Termal Profil (noktalı çizgi = optimal sıcaklık)')
ax.legend()
ax.grid(alpha=0.3)

# Grip mu
ax = axes[1, 0]
for comp, df in compound_data.items():
    ax.plot(df['lap'], df['grip_mu'], color=colors[comp], linewidth=2.5, label=comp.capitalize())
ax.set_xlabel('Tur')
ax.set_ylabel('Efektif grip μ')
ax.set_title('Grip Katsayısı Evrim')
ax.legend()
ax.grid(alpha=0.3)

# Kümülatif tur süresi kaybı
ax = axes[1, 1]
for comp, df in compound_data.items():
    cumulative = df['lap_delta'].cumsum()
    ax.plot(df['lap'], cumulative, color=colors[comp], linewidth=2.5, label=comp.capitalize())
ax.set_xlabel('Tur')
ax.set_ylabel('Toplam kaybedilen süre (s)')
ax.set_title('Kümülatif Tur Süresi Kaybı')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_stint_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

# Sayısal özet
print("=== Stint Özeti (45 tur) ===")
for comp, df in compound_data.items():
    total = df['lap_delta'].sum()
    cliff = df[df['past_cliff']]['lap'].min() if df['past_cliff'].any() else 'Yok'
    print(f"  {comp:6s}: Toplam kayıp={total:.2f}s | Cliff tur={cliff}")


## 3. Undercut / Overcut Analizi

In [ ]:
# Undercut fırsat haritası: gap vs fresh tyre gain
from pitstop.simulation.tire_model import undercut_delta

gap_range = np.linspace(0.2, 5.0, 30)
gain_range = np.linspace(0.1, 2.0, 30)
pit_loss = 22.5  # typical pit loss in seconds

uc_map = np.zeros((len(gain_range), len(gap_range)))
for i, gain in enumerate(gain_range):
    for j, gap in enumerate(gap_range):
        result = undercut_delta(gap, pit_loss, gain, laps_to_respond=1)
        uc_map[i, j] = result['undercut_delta']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
ax = axes[0]
im = ax.contourf(gap_range, gain_range, uc_map, levels=20, cmap='RdYlGn')
ax.contour(gap_range, gain_range, uc_map, levels=[0], colors='black', linewidths=2)
plt.colorbar(im, ax=ax, label='Undercut delta (s)')
ax.set_xlabel('Öndeki araca gap (s)')
ax.set_ylabel('Fresh lastik kazancı (s/tur)')
ax.set_title('Undercut Fırsat Haritası\n(siyah çizgi = breakeven)', fontweight='bold')
ax.text(0.5, 1.5, 'UNDERCUT ✓', fontsize=12, color='darkgreen', fontweight='bold')
ax.text(3.0, 0.3, 'OVERCUT / BEKLE', fontsize=10, color='darkred')

# Sensitivity: gap değişirken undercut delta
ax2 = axes[1]
for gain in [0.3, 0.6, 0.9, 1.2, 1.5]:
    deltas = [undercut_delta(g, pit_loss, gain)['undercut_delta'] for g in gap_range]
    ax2.plot(gap_range, deltas, linewidth=2, label=f'Gain={gain}s/lap')
ax2.axhline(0, color='black', linewidth=1.5, linestyle='--')
ax2.fill_between(gap_range, 0, 10, alpha=0.05, color='green')
ax2.fill_between(gap_range, -30, 0, alpha=0.05, color='red')
ax2.set_xlabel('Gap (s)')
ax2.set_ylabel('Undercut delta (s)')
ax2.set_title('Undercut Delta vs Gap\n(farklı fresh tyre gain değerleri)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(-25, 5)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('02_undercut_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
